# NER with RoBERTa + CRF

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers pytorch-crf seqeval

In [ ]:
# Cell 2: Imports
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
import numpy as np
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Config
MODEL_NAME = 'roberta-base'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 3e-5
DROPOUT = 0.1

In [ ]:
# Cell 4: Locate and read CoNLL files
# Works on Colab and locally without manual path editing.
#   Local: walks up from cwd to find data/processed/ in the project tree.
#   Colab: tries /content/data/processed, /content/drive/MyDrive/NER/data/processed,
#          then /content/ (bare files) - does NOT auto-mount Drive.
IN_COLAB = 'google.colab' in sys.modules

def resolve_data_dir():
    candidates = []
    if IN_COLAB:
        candidates += [
            '/content/data/processed',
            '/content/drive/MyDrive/NER/data/processed',
            '/content',
        ]
    cur = os.getcwd()
    for _ in range(6):
        candidates.append(os.path.join(cur, 'data', 'processed'))
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent

    for d in candidates:
        if os.path.exists(os.path.join(d, 'train.conll')):
            return os.path.abspath(d)

    msg = ['Could not find train.conll. Tried:'] + ['  ' + os.path.abspath(c) for c in candidates]
    if IN_COLAB:
        msg += [
            '',
            'On Colab, upload the three .conll files into /content/ and re-run this cell:',
            '    from google.colab import files',
            '    files.upload()  # select train.conll, valid.conll, test.conll',
        ]
    raise FileNotFoundError('\n'.join(msg))

def read_conll(filepath):
    """Read CoNLL file, return list of (tokens, tags) per document."""
    sentences, tokens, tags = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    tokens.append(parts[0])
                    tags.append(parts[1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

DATA_DIR = resolve_data_dir()
print(f'DATA_DIR: {DATA_DIR}')

train_data = read_conll(os.path.join(DATA_DIR, 'train.conll'))
valid_data = read_conll(os.path.join(DATA_DIR, 'valid.conll'))
test_data  = read_conll(os.path.join(DATA_DIR, 'test.conll'))

print(f'Train: {len(train_data)} docs')
print(f'Valid: {len(valid_data)} docs')
print(f'Test:  {len(test_data)} docs')

In [ ]:
# Cell 5: Build label mapping
all_tags = set()
for tokens, tags in train_data + valid_data + test_data:
    all_tags.update(tags)

label_list = sorted(all_tags)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(label_list)

print(f'Labels ({num_labels}): {label_list}')

In [ ]:
# Cell 6: Dataset class
# RoBERTa uses BPE; add_prefix_space=True is required so that encoding a bare
# word produces the same tokens it would receive mid-sentence. Without it,
# word-level NER alignment silently breaks.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)

class NERDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        # Truncate to max_len - 2 (for <s> and </s>)
        tokens = tokens[:self.max_len - 2]
        tags = tags[:self.max_len - 2]

        # Tokenize word by word, align labels to subwords
        input_ids = [self.tokenizer.cls_token_id]
        label_ids = [self.label2id['O']]  # <s> gets O

        for word, tag in zip(tokens, tags):
            word_tokens = self.tokenizer.encode(word, add_special_tokens=False)
            if not word_tokens:
                continue
            # Only first subword gets the real label, rest get same label
            input_ids.extend(word_tokens)
            label_ids.append(self.label2id[tag])
            # For extra subword tokens, use same tag (I- if B-, else same)
            for _ in range(1, len(word_tokens)):
                if tag.startswith('B-'):
                    label_ids.append(self.label2id['I-' + tag[2:]])
                else:
                    label_ids.append(self.label2id[tag])

        # Truncate if subword expansion exceeded max_len
        input_ids = input_ids[:self.max_len - 1]
        label_ids = label_ids[:self.max_len - 1]

        # Add </s>
        input_ids.append(self.tokenizer.sep_token_id)
        label_ids.append(self.label2id['O'])

        # Padding
        seq_len = len(input_ids)
        pad_len = self.max_len - seq_len
        attention_mask = [1] * seq_len + [0] * pad_len
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        label_ids += [self.label2id['O']] * pad_len

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'seq_len': seq_len
        }

train_ds = NERDataset(train_data, tokenizer, label2id, MAX_LEN)
valid_ds = NERDataset(valid_data, tokenizer, label2id, MAX_LEN)
test_ds  = NERDataset(test_data,  tokenizer, label2id, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)}, Valid: {len(valid_loader)}, Test: {len(test_loader)}')

In [ ]:
# Cell 7: RoBERTa + CRF Model
# Note: the loader may report `lm_head.*` keys as UNEXPECTED - that's the
# pretraining MLM head we're not using. Safe to ignore (same as BERT's `cls.*`).
class RoBERTa_CRF(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        emissions = self.classifier(self.dropout(outputs.last_hidden_state))
        mask = attention_mask.bool()

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)

model = RoBERTa_CRF(MODEL_NAME, num_labels, DROPOUT).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 8: Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

In [ ]:
# Cell 9: Evaluation function
def evaluate(model, dataloader, id2label):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            seq_lens = batch['seq_len']

            preds = model(input_ids, attention_mask)

            for i in range(len(preds)):
                slen = seq_lens[i].item()
                # Skip <s> and </s>: indices 1 to slen-2
                true_tags = [id2label[labels[i][j].item()] for j in range(1, slen - 1)]
                pred_tags = [id2label[preds[i][j]] for j in range(1, slen - 1)]
                y_true.append(true_tags)
                y_pred.append(pred_tags)

    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    f = f1_score(y_true, y_pred)
    return p, r, f, y_true, y_pred

In [ ]:
# Cell 10: Training loop
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        loss = model(input_ids, attention_mask, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} Step {step+1}/{len(train_loader)} Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    p, r, f1, _, _ = evaluate(model, valid_loader, id2label)
    print(f'Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.4f} | Val P: {p:.4f} R: {r:.4f} F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_roberta_crf.pt')
        print(f'  -> Saved best model (F1: {f1:.4f})')

print(f'\nTraining complete. Best Val F1: {best_f1:.4f}')

In [ ]:
# Cell 11: Test evaluation
model.load_state_dict(torch.load('best_roberta_crf.pt'))
p, r, f1, y_true, y_pred = evaluate(model, test_loader, id2label)

print('='*60)
print('TEST RESULTS - RoBERTa + CRF')
print('='*60)
print(f'Precision: {p:.4f}')
print(f'Recall:    {r:.4f}')
print(f'F1-Score:  {f1:.4f}')
print('='*60)
print()
print(classification_report(y_true, y_pred))

In [ ]:
# Cell 12 (Optional): Save model for deployment
import json

os.makedirs('saved_model_roberta', exist_ok=True)
torch.save(model.state_dict(), 'saved_model_roberta/model.pt')
with open('saved_model_roberta/labels.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f)
tokenizer.save_pretrained('saved_model_roberta')
print('Model saved to saved_model_roberta/')